# Sesión 1. Formatos de datos genómicos y control de calidad

En esta sesión vamos a tomar datos genómicos reales de 969 personas de 15 poblaciones, entender en qué formato vienen y dejarlos listos para analizar la ancestría. Es la base de todo lo que sigue: sin control de calidad, los análisis de ancestría dan resultados engañosos.

Pasos:

1. Descargar PLINK 2
2. Descargar los datos y mirar el formato VCF
3. Convertir al formato binario de PLINK
4. Control de calidad y poda por LD

Después de cada paso hay una tabla con los archivos que crea, qué son y si se usan más adelante. El resumen completo, con un diagrama de cómo se conectan, está en [archivos_por_paso.md](https://github.com/YTZ27/curso-ancestria/blob/main/notebooks/archivos_por_paso.md).

**No hay que instalar nada:** todo corre en este cuaderno, en los computadores de Google. Para ejecutar una celda, haz clic en el botón de reproducir que aparece a su izquierda, o presiona Shift + Enter.

## 1. Descargar PLINK 2

PLINK es el programa más usado para manipular datos genómicos. Lo descargamos y comprobamos que quedó funcionando.

Si el enlace deja de funcionar, entra a la [página de PLINK 2](https://www.cog-genomics.org/plink/2.0/), copia el enlace de **Linux 64-bit Intel** de la columna **Alpha** y pégalo en la línea del `wget`. El resto de la celda no cambia.

In [ ]:
!wget -nv -O plink2.zip https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20261015.zip
!unzip -o plink2.zip
!./plink2 --version
!ls -lh plink2.zip plink2

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `plink2.zip` | La descarga comprimida | No: solo sirve para extraer `plink2` |
| `plink2` | El programa PLINK 2 | Sí: en los pasos 3 y 4 |

## 2. Descargar los datos y mirar el formato VCF

El archivo `exome.vcf.gz` está en el repositorio del curso. Contiene personas del Proyecto 1000 Genomas y del Human Genome Diversity Panel (HGDP), restringidas a las regiones codificantes: los archivos VCF originales ([recursos de PLINK 2](https://www.cog-genomics.org/plink/2.0/resources)) se filtraron con el archivo BED del kit de exoma [Agilent SureSelectXT Human All Exon V8](https://www.agilent.com/en/product/next-generation-sequencing/ngs-assays-panels/exome/sureselect-human-all-exon-4252439).

In [ ]:
!wget -nv -nc https://github.com/YTZ27/curso-ancestria/raw/main/data/exome.vcf.gz
!ls -lh exome.vcf.gz

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `exome.vcf.gz` | Genotipos de 969 personas en 244 305 variantes, en formato VCF (una tabla de texto) comprimido con gzip (`.gz`) | Sí: es la entrada del paso 3 |

**El formato VCF.** Las líneas que empiezan con `##` son el encabezado; después viene una tabla con una fila por variante. Veamos las 3 primeras variantes y las 3 primeras personas.

In [ ]:
!zcat exome.vcf.gz | grep -v '^##' | head -4 | cut -f1-12

Las columnas son: `#CHROM` (cromosoma), `POS` (posición), `ID`, `REF` (alelo de referencia), `ALT` (alelo alternativo), `QUAL`, `FILTER`, `INFO`, `FORMAT`, y luego **una columna por persona**.

El genotipo de cada persona se escribe con dos números separados por `/`:

| Genotipo | Significa |
|---|---|
| `0/0` | Dos alelos de referencia (homocigoto de referencia) |
| `0/1` | Uno de cada (heterocigoto) |
| `1/1` | Dos alelos alternativos (homocigoto alternativo) |
| `./.` | Dato faltante: no se pudo genotipar |

¿Cuántas personas y cuántas variantes hay?

In [ ]:
!echo "Personas:  $(zcat exome.vcf.gz 2>/dev/null | grep -m1 '^#CHROM' | cut -f10- | wc -w)"
!echo "Variantes: $(zcat exome.vcf.gz | grep -vc '^#')"

**¿De dónde viene cada persona?** El identificador empieza con el nombre de su población. Las poblaciones que llevan `forReference` son las que usaremos como referencia de cada continente.

In [ ]:
import pandas as pd

ids = !zcat exome.vcf.gz 2>/dev/null | grep -m1 '^#CHROM' | cut -f10- | tr '\t' '\n'
poblacion = pd.Series([i.split('_')[0] for i in ids if i])
poblacion.value_counts().sort_index()

## 3. Convertir al formato binario de PLINK

El VCF es texto y ocupa mucho. PLINK guarda cada genotipo en solo 2 bits, así que sus archivos se procesan rápido, y ADMIXTURE (que usaremos en la sesión 2) los lee directamente.

La conversión crea tres archivos que **siempre van juntos**: `.bed` (genotipos), `.bim` (variantes) y `.fam` (personas).

In [ ]:
!./plink2 --vcf exome.vcf.gz --make-bed --out exome_data
!ls -lh exome_data.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `exome_data.bed` | Genotipos en formato binario. No se puede leer como texto: no lo abras | Sí: entrada del paso 4 |
| `exome_data.bim` | Una fila por variante: cromosoma, identificador, posición genética (0 = desconocida), posición en pares de bases, alelo 1, alelo 2 | Sí: entrada del paso 4 |
| `exome_data.fam` | Una fila por persona: identificador de familia, identificador de la persona, padre, madre, sexo (0 = desconocido), fenotipo (-9 = faltante) | Sí: entrada del paso 4 |
| `exome_data.log` | Registro del comando y resumen del resultado | No, pero es donde se revisa qué hizo PLINK |

PLINK lee los tres archivos por su nombre común (prefijo): `--bfile exome_data` significa `exome_data.bed` + `exome_data.bim` + `exome_data.fam`. El `.bed` de PLINK **no** es lo mismo que un archivo BED de regiones genómicas; son formatos distintos con el mismo nombre.

Las primeras filas del `.bim` (variantes) y del `.fam` (personas):

In [ ]:
!head -3 exome_data.bim
!head -3 exome_data.fam

## 4. Control de calidad y poda por LD

Antes de estimar ancestría hay que quitar las variantes y las personas poco confiables. Aplicamos un control de calidad estándar que conserva las variantes:

| Filtro | Qué conserva | Por qué |
|---|---|---|
| `--maf 0.01` | Variantes con frecuencia de al menos 1 % | Las muy raras aportan poco a la ancestría y suelen ser errores |
| `--geno 0.1` | Variantes genotipadas en al menos el 90 % de las personas | Si falta mucho dato, la variante no es confiable |
| `--mind 0.1` | Personas con al menos el 90 % de sus variantes genotipadas | Una muestra con mucho dato faltante suele ser de mala calidad |

In [ ]:
!./plink2 --bfile exome_data \
       --maf 0.01 \
       --geno 0.1 \
       --mind 0.1 \
       --make-bed \
       --out qc_maf-0.01_geno-0.1
!ls -lh qc_maf-0.01_geno-0.1.*
!grep -E "removed|remaining" qc_maf-0.01_geno-0.1.log

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `qc_maf-0.01_geno-0.1.bed / .bim / .fam` | Los mismos datos, conservando solo las variantes y personas que pasaron los filtros. El nombre indica los filtros usados | Sí: entrada de la poda por LD (pasos 4.1 y 4.2) |
| `qc_maf-0.01_geno-0.1.log` | Resumen de los filtros | No. La última línea de la celda muestra sus líneas clave: cuántas variantes y personas eliminó cada filtro |

Con los datos del curso no se elimina ninguna persona, 133 675 variantes tienen frecuencia menor al 1 % y quedan 110 630 variantes.

### ¿Por qué no filtramos por equilibrio de Hardy-Weinberg?

En un control de calidad típico también se eliminan las variantes que se desvían del **equilibrio de Hardy-Weinberg** (`--hwe`), porque una desviación fuerte suele indicar un error de genotipado (Anderson et al., 2010).

Aquí no lo hacemos, y la razón es el objetivo del análisis. El equilibrio de Hardy-Weinberg supone **una sola población que se cruza al azar**. Nuestros datos son 15 poblaciones juntas, y cuando se mezclan poblaciones con frecuencias alélicas distintas, el conjunto muestra menos heterocigotos de los esperados aunque cada población por separado esté en equilibrio. Es el **efecto Wahlund** (Hartl y Clark, 2007).

El problema es que ese efecto es más fuerte justamente en las variantes que más difieren entre continentes, que son las informativas de ancestría. Con estos datos, el filtro `--hwe 0.0001` elimina el **81,5 %** de las variantes cuya frecuencia difiere en más de 0,5 entre continentes. En las variantes que eliminaba, la mediana de esa diferencia era 0,45; en las que conservaba, 0,08.

Dicho de otro modo: el filtro estaba quitando la señal que queremos medir.

**Qué hacer en un análisis propio:** si quieres filtrar por Hardy-Weinberg, aplícalo **dentro de cada población por separado** (con `--keep`) y elimina solo las variantes que fallan en varias poblaciones a la vez. Si el objetivo es estimar ancestría, omitirlo es una opción razonable.

### Poda por desequilibrio de ligamiento (LD)

Las variantes cercanas en el genoma se heredan juntas y repiten la misma información. El PCA y ADMIXTURE suponen que las variantes son independientes, así que hay que quedarse con un subconjunto que lo sea aproximadamente (Alexander et al., 2009).

Se hace en dos pasos:

* Ventanas de 50 variantes, avanzando de 5 en 5 (`50 5`)
* En cada ventana eliminamos una variante de cada par con r² > 0,2

**Paso 4.1:** generar la lista de variantes a conservar.

In [ ]:
!./plink2 --bfile qc_maf-0.01_geno-0.1 \
       --indep-pairwise 50 5 0.2 \
       --out ld_pruned
!wc -l ld_pruned.prune.in ld_pruned.prune.out

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `ld_pruned.prune.in` | Lista de identificadores de las variantes que se conservan (aproximadamente independientes entre sí) | Sí: en el paso 4.2 |
| `ld_pruned.prune.out` | Lista de identificadores eliminados por estar en LD con una variante que se conserva | No: solo para revisar |
| `ld_pruned.log` | Registro del comando | No |

Este paso no modifica los datos: solo escribe listas. Con los datos del curso se conservan 59 850 variantes y se eliminan 50 780.

**Paso 4.2:** conservar solo las variantes de `ld_pruned.prune.in`.

In [ ]:
!./plink2 --bfile qc_maf-0.01_geno-0.1 \
       --extract ld_pruned.prune.in \
       --make-bed \
       --out admixture_LD_QC
!ls -lh admixture_LD_QC.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `admixture_LD_QC.bed / .bim / .fam` | Datos finales: pasaron el control de calidad y la poda por LD (969 personas, 59 850 variantes) | Sí: es el punto de partida de la sesión 2 |
| `admixture_LD_QC.log` | Registro del comando | No |

## Resumen de la sesión

Partimos de un VCF con 244 305 variantes y terminamos con 59 850 variantes en formato binario de PLINK, listas para estimar ancestría.

| Paso | Variantes que quedan |
|---|---|
| Datos originales | 244 305 |
| Después del control de calidad | 110 630 |
| Después de la poda por LD | 59 850 |

**Importante:** cuando cierres este cuaderno, Colab borra todos los archivos. No hay que guardarlos: el cuaderno de la sesión 2 vuelve a generarlos en una sola celda, en menos de un minuto.

## Referencias

- Alexander DH, Novembre J, Lange K. Fast model-based estimation of ancestry in unrelated individuals. *Genome Res*. 2009;19:1655–1664. doi:10.1101/gr.094052.109
- Anderson CA, et al. Data quality control in genetic case-control association studies. *Nat Protoc*. 2010;5:1564–1573. doi:10.1038/nprot.2010.116
- Chang CC, et al. Second-generation PLINK: rising to the challenge of larger and richer datasets. *GigaScience*. 2015;4:7. doi:10.1186/s13742-015-0047-8
- Danecek P, et al. The variant call format and VCFtools. *Bioinformatics*. 2011;27:2156–2158. doi:10.1093/bioinformatics/btr330
- Hartl DL, Clark AG. *Principles of Population Genetics*. 4.ª ed. Sinauer; 2007 (efecto Wahlund)
- 1000 Genomes Project Consortium. A global reference for human genetic variation. *Nature*. 2015;526:68–74. doi:10.1038/nature15393